# QuantFormer - Notebook 08
# Training the Multimodal Fusion Model

---

## Objective

In this notebook, we train the final multimodal fusion model of QuantFormer.

The pretrained Temporal Fusion Transformer (TFT) extracts market representations from the FI-2010 Limit Order Book dataset, while FinBERT extracts semantic embeddings from financial news headlines. These two modalities are fused through a lightweight neural network to predict future market movement.

### Pipeline

FI-2010 Market Data
        │
        ▼
Pretrained TFT
        │
        ▼
128-D Market Features

Financial News
        │
        ▼
Pretrained FinBERT
        │
        ▼
768-D News Embeddings

        │
        ▼
Multimodal Fusion Network
        │
        ▼
DOWN / STABLE / UP

---

### Notebook Goals

- Load FI-2010 training dataset
- Load pretrained Temporal Fusion Transformer
- Load pretrained FinBERT
- Generate multimodal features
- Train Fusion Network
- Evaluate model performance
- Save best fusion checkpoint

In [ ]:
import os
import sys
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

c:\Users\gupta\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("=" * 60)
print("Random Seed Fixed")
print("=" * 60)

print("Seed :", SEED)

Random Seed Fixed
Seed : 42


In [ ]:
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 60)
print("Device Configuration")
print("=" * 60)

print("Using Device :", DEVICE)

if DEVICE.type == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))

Device Configuration
Using Device : cpu


In [ ]:
PROJECT_ROOT = Path.cwd().parent

print("=" * 60)
print("Project Root")
print("=" * 60)

print(PROJECT_ROOT)

Project Root
d:\Coding\Quant Former


In [ ]:
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Project added to sys.path")

Project added to sys.path


In [ ]:
DATASET_DIR = PROJECT_ROOT / "datasets"
PROCESSED_DIR = DATASET_DIR / "processed" / "FI2010"

CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"

CHECKPOINT_DIR.mkdir(exist_ok=True)
ARTIFACT_DIR.mkdir(exist_ok=True)

print("=" * 60)
print("Project Directories")
print("=" * 60)

print("Datasets   :", DATASET_DIR)
print("Processed  :", PROCESSED_DIR)
print("Checkpoints:", CHECKPOINT_DIR)
print("Artifacts  :", ARTIFACT_DIR)

Project Directories
Datasets   : d:\Coding\Quant Former\datasets
Processed  : d:\Coding\Quant Former\datasets\processed\FI2010
Checkpoints: d:\Coding\Quant Former\checkpoints
Artifacts  : d:\Coding\Quant Former\artifacts


In [ ]:
required_files = [
    PROCESSED_DIR / "X_train.npy",
    PROCESSED_DIR / "y_train.npy",
    CHECKPOINT_DIR / "best_tft_model.pth",
]

print("=" * 60)
print("Verifying Project Files")
print("=" * 60)

for file in required_files:
    status = "FOUND" if file.exists() else "MISSING"
    print(f"{status:8} : {file.name}")

Verifying Project Files
FOUND    : X_train.npy
FOUND    : y_train.npy
FOUND    : best_tft_model.pth


## Observation

The project environment has been initialized successfully.

- Random seed fixed for reproducibility.
- Device configuration verified.
- Project directory structure detected.
- Required datasets and pretrained TFT checkpoint are available.
- The notebook is ready to begin loading the training dataset in the next section.

# Section 2 — Load FI-2010 Training Dataset

---

## Objective

In this section, we load the processed FI-2010 training dataset used for training the multimodal fusion model.

The dataset contains preprocessed sequences generated during earlier preprocessing stages.

Each training sample consists of:

- Sequence Length = 100
- Features = 143
- Target Label = {DOWN, STABLE, UP}

After loading the dataset, a custom PyTorch Dataset and DataLoader are created for efficient batch-wise training.

In [ ]:
from src.data.dataset import FI2010Dataset

In [ ]:
X_train = np.load(
    PROCESSED_DIR / "X_train.npy"
)

y_train = np.load(
    PROCESSED_DIR / "y_train.npy"
)

print("=" * 60)
print("FI-2010 Training Dataset Loaded")
print("=" * 60)

print("X_train Shape :", X_train.shape)
print("y_train Shape :", y_train.shape)

FI-2010 Training Dataset Loaded
X_train Shape : (289821, 100, 143)
y_train Shape : (289821,)


In [ ]:
train_dataset = FI2010Dataset(
    X_train,
    y_train
)

print("=" * 60)
print("Training Dataset Created")
print("=" * 60)

print("Dataset Size :", len(train_dataset))

Training Dataset Created
Dataset Size : 289821


In [ ]:
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

print("=" * 60)
print("Training DataLoader Ready")
print("=" * 60)

print("Batch Size :", BATCH_SIZE)
print("Number of Batches :", len(train_loader))

Training DataLoader Ready
Batch Size : 64
Number of Batches : 4529


In [ ]:
features, labels = next(iter(train_loader))

print("=" * 60)
print("One Training Batch")
print("=" * 60)

print("Features Shape :", features.shape)
print("Labels Shape   :", labels.shape)

print()
print("Features Device :", features.device)
print("Labels Device   :", labels.device)

One Training Batch
Features Shape : torch.Size([64, 100, 143])
Labels Shape   : torch.Size([64])

Features Device : cpu
Labels Device   : cpu


In [ ]:
unique_labels, counts = np.unique(
    y_train,
    return_counts=True
)

class_names = {
    0: "DOWN",
    1: "STABLE",
    2: "UP"
}

print("=" * 60)
print("Training Dataset Statistics")
print("=" * 60)

for label, count in zip(unique_labels, counts):
    print(f"{class_names[int(label)]:8} : {count}")

Training Dataset Statistics
DOWN     : 115115
STABLE   : 62396
UP       : 112310


In [ ]:
sample_features, sample_label = train_dataset[0]

print("=" * 60)
print("First Training Sample")
print("=" * 60)

print("Feature Shape :", sample_features.shape)
print("Label :", class_names[int(sample_label)])

First Training Sample
Feature Shape : torch.Size([100, 143])
Label : DOWN


## Observation

The FI-2010 training dataset has been successfully loaded.

Key observations:

- Processed NumPy arrays were loaded without errors.
- A custom PyTorch Dataset object was created.
- Training DataLoader was initialized with shuffled mini-batches.
- Each batch contains 64 market sequences of shape (100, 143).
- The dataset is now ready for feature extraction using the pretrained Temporal Fusion Transformer in the next section.

# Section 3 — Load Pretrained Temporal Fusion Transformer

---

## Objective

The Temporal Fusion Transformer (TFT) has already been trained on the FI-2010 Limit Order Book dataset in the previous notebook.

In this notebook, the TFT acts as a **fixed market feature extractor**.

Instead of retraining the TFT, we:

- Load the pretrained checkpoint
- Restore learned weights
- Freeze all model parameters
- Switch the model to evaluation mode

The output of the TFT is a **128-dimensional market representation**, which will later be fused with FinBERT news embeddings.

In [ ]:
from src.models.tft_model import TemporalFusionTransformer

In [ ]:
tft_model = TemporalFusionTransformer()

print("=" * 60)
print("Temporal Fusion Transformer Created")
print("=" * 60)

print(type(tft_model))

Temporal Fusion Transformer Created
<class 'src.models.tft_model.TemporalFusionTransformer'>


In [ ]:
TFT_CHECKPOINT = CHECKPOINT_DIR / "best_tft_model.pth"

checkpoint = torch.load(
    TFT_CHECKPOINT,
    map_location=DEVICE
)

print("=" * 60)
print("Checkpoint Loaded")
print("=" * 60)

print(checkpoint.keys())

Checkpoint Loaded
dict_keys(['epoch', 'model_state_dict', 'optimizer_state_dict', 'best_validation_accuracy'])


In [ ]:
tft_model.load_state_dict(
    checkpoint["model_state_dict"]
)

print("=" * 60)
print("Pretrained TFT Weights Loaded Successfully")
print("=" * 60)

Pretrained TFT Weights Loaded Successfully


In [ ]:
tft_model = tft_model.to(DEVICE)

print("=" * 60)
print("TFT moved to", DEVICE)
print("=" * 60)

TFT moved to cpu


In [ ]:
for parameter in tft_model.parameters():
    parameter.requires_grad = False

print("=" * 60)
print("All TFT Parameters Frozen")
print("=" * 60)

All TFT Parameters Frozen


In [ ]:
tft_model.eval()

print("=" * 60)
print("TFT set to Evaluation Mode")
print("=" * 60)

TFT set to Evaluation Mode


In [ ]:
total_params = sum(
    p.numel()
    for p in tft_model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in tft_model.parameters()
    if p.requires_grad
)

print("=" * 60)
print("Temporal Fusion Transformer Summary")
print("=" * 60)

print(f"Total Parameters     : {total_params:,}")
print(f"Trainable Parameters : {trainable_params:,}")

Temporal Fusion Transformer Summary
Total Parameters     : 654,723
Trainable Parameters : 0


In [ ]:
sample_features, _ = next(iter(train_loader))

sample_features = sample_features.to(DEVICE)

with torch.no_grad():
    logits, market_features, attention = tft_model(sample_features)

print("=" * 60)
print("TFT Forward Pass Successful")
print("=" * 60)

print("Classification Logits :", logits.shape)
print("Market Features       :", market_features.shape)
print("Attention Weights     :", attention.shape)

TFT Forward Pass Successful
Classification Logits : torch.Size([64, 3])
Market Features       : torch.Size([64, 128])
Attention Weights     : torch.Size([64, 4, 100, 100])


In [ ]:
print("=" * 60)
print("Pretrained TFT Information")
print("=" * 60)

print("Checkpoint :", TFT_CHECKPOINT.name)
print("Best Validation Accuracy :", checkpoint["best_validation_accuracy"])
print("Epoch :", checkpoint["epoch"])

Pretrained TFT Information
Checkpoint : best_tft_model.pth
Best Validation Accuracy : 0.8375816858015225
Epoch : 15


## Observation

The pretrained Temporal Fusion Transformer was successfully restored from the saved checkpoint.

Key observations:

- The checkpoint weights were loaded correctly.
- All TFT parameters were frozen to preserve the learned market representation.
- The model was switched to evaluation mode.
- A forward pass confirmed that the TFT generates a **128-dimensional market feature vector** along with classification logits and attention weights.
- These market features will serve as one branch of the multimodal fusion network in the upcoming sections.

# Section 4 — Load Pretrained FinBERT

---

## Objective

Financial news contains valuable sentiment information that influences market movements.

In this section, we load the pretrained **FinBERT** model, which has been trained specifically for financial text understanding.

The FinBERT model is used only as a **feature extractor**.

For every financial headline:

Financial Headline
        │
        ▼
Tokenizer
        │
        ▼
FinBERT
        │
        ▼
768-Dimensional Embedding

These embeddings will later be fused with the 128-dimensional market features extracted by the Temporal Fusion Transformer.

To preserve the pretrained knowledge, all FinBERT parameters remain frozen during training.

In [ ]:
from transformers import AutoTokenizer, AutoModel

In [ ]:
MODEL_NAME = "ProsusAI/finbert"

print("=" * 60)
print("FinBERT Model")
print("=" * 60)
print(MODEL_NAME)

FinBERT Model
ProsusAI/finbert


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("=" * 60)
print("Tokenizer Loaded Successfully")
print("=" * 60)

Tokenizer Loaded Successfully


In [ ]:
finbert_model = AutoModel.from_pretrained(
    MODEL_NAME
)

print("=" * 60)
print("FinBERT Loaded Successfully")
print("=" * 60)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 22100.42it/s]
[transformers] BertModel LOAD REPORT from: ProsusAI/finbert
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FinBERT Loaded Successfully


In [ ]:
finbert_model = finbert_model.to(DEVICE)

print("=" * 60)
print("FinBERT moved to", DEVICE)
print("=" * 60)

FinBERT moved to cpu


In [ ]:
for parameter in finbert_model.parameters():
    parameter.requires_grad = False

print("=" * 60)
print("FinBERT Parameters Frozen")
print("=" * 60)

FinBERT Parameters Frozen


In [ ]:
finbert_model.eval()

print("=" * 60)
print("FinBERT set to Evaluation Mode")
print("=" * 60)

FinBERT set to Evaluation Mode


In [ ]:
total_params = sum(
    p.numel()
    for p in finbert_model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in finbert_model.parameters()
    if p.requires_grad
)

print("=" * 60)
print("FinBERT Summary")
print("=" * 60)

print(f"Total Parameters     : {total_params:,}")
print(f"Trainable Parameters : {trainable_params:,}")

FinBERT Summary
Total Parameters     : 109,482,240
Trainable Parameters : 0


In [ ]:
sample_headline = (
    "Apple reports strong quarterly earnings driven by AI demand."
)

inputs = tokenizer(
    sample_headline,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=128
)

inputs = {
    key: value.to(DEVICE)
    for key, value in inputs.items()
}

with torch.no_grad():

    outputs = finbert_model(**inputs)

embedding = outputs.last_hidden_state.mean(dim=1)

print("=" * 60)
print("Sample News Embedding")
print("=" * 60)

print("Embedding Shape :", embedding.shape)

Sample News Embedding
Embedding Shape : torch.Size([1, 768])


In [ ]:
print("=" * 60)
print("Embedding Statistics")
print("=" * 60)

print("Mean :", embedding.mean().item())
print("Std  :", embedding.std().item())
print("Min  :", embedding.min().item())
print("Max  :", embedding.max().item())

Embedding Statistics
Mean : -0.01635274663567543
Std  : 0.4366307556629181
Min  : -1.4561094045639038
Max  : 1.503316879272461


In [ ]:
print("=" * 60)
print("First 20 Embedding Values")
print("=" * 60)

print(embedding[0][:20])

First 20 Embedding Values
tensor([ 0.0061,  0.0774,  0.1646, -0.0646,  0.5067, -0.0194,  0.0739, -0.0958,
         0.1617, -0.3284, -0.1081, -0.2244,  0.8183, -0.4908,  0.1811,  0.1997,
         0.2272,  0.0216, -0.7466,  0.5054])


## Observation

The pretrained FinBERT model was successfully loaded and initialized.

Key observations:

- The tokenizer correctly processed financial text.
- FinBERT generated a 768-dimensional semantic embedding for the sample headline.
- All model parameters were frozen, ensuring that FinBERT acts solely as a feature extractor.
- These news embeddings will be combined with the 128-dimensional market features extracted by the pretrained Temporal Fusion Transformer in the upcoming fusion training stage.

# Section 5 — Load Financial News Dataset

---

## Objective

The QuantFormer fusion model combines market microstructure information with financial news sentiment.

In this section, we load the **FinancialPhraseBank** dataset, which contains manually annotated financial headlines.

Each headline is later converted into a **768-dimensional FinBERT embedding** and paired with market features extracted by the Temporal Fusion Transformer.

### News Processing Pipeline

FinancialPhraseBank
        │
        ▼
Financial Headlines
        │
        ▼
News Dataset
        │
        ▼
DataLoader
        │
        ▼
FinBERT
        │
        ▼
768-D News Embeddings

In [ ]:
NEWS_DATA_PATH = (
    PROJECT_ROOT
    / "datasets"
    / "FinancialPhraseBank-v1.0"
    / "Sentences_50Agree.txt"
)

print("=" * 60)
print("FinancialPhraseBank Dataset")
print("=" * 60)

print(NEWS_DATA_PATH)
print()
print("Exists :", NEWS_DATA_PATH.exists())

FinancialPhraseBank Dataset
d:\Coding\Quant Former\datasets\FinancialPhraseBank-v1.0\Sentences_50Agree.txt

Exists : True


In [ ]:
news_data = []

with open(
    NEWS_DATA_PATH,
    "r",
    encoding="latin-1"
) as file:

    for line in file:

        line = line.strip()

        if not line:
            continue

        if "@" in line:

            headline, sentiment = line.rsplit("@", 1)

            news_data.append(
                {
                    "headline": headline.strip(),
                    "sentiment": sentiment.strip()
                }
            )

print("=" * 60)
print("FinancialPhraseBank Loaded Successfully")
print("=" * 60)

print("Total Headlines :", len(news_data))

FinancialPhraseBank Loaded Successfully
Total Headlines : 4846


In [ ]:
print("=" * 60)
print("Sample Headlines")
print("=" * 60)

for i in range(5):

    print(f"Headline {i+1}")

    print(news_data[i]["headline"])

    print("Sentiment :", news_data[i]["sentiment"])

    print("" + "-" * 60)

Sample Headlines
Headline 1
According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing .
Sentiment : neutral
------------------------------------------------------------
Headline 2
Technopolis plans to develop in stages an area of no less than 100,000 square meters in order to host companies working in computer technologies and telecommunications , the statement said .
Sentiment : neutral
------------------------------------------------------------
Headline 3
The international electronic industry company Elcoteq has laid off tens of employees from its Tallinn facility ; contrary to earlier layoffs the company contracted the ranks of its office workers , the daily Postimees reported .
Sentiment : negative
------------------------------------------------------------
Headline 4
With the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw mate

In [ ]:
class FinancialNewsDataset(Dataset):

    def __init__(self, news):

        self.news = news

    def __len__(self):

        return len(self.news)

    def __getitem__(self, index):

        sample = self.news[index]

        return (
            sample["headline"],
            sample["sentiment"]
        )

In [ ]:
news_dataset = FinancialNewsDataset(
    news_data
)

print("=" * 60)
print("News Dataset Created")
print("=" * 60)

print("Dataset Size :", len(news_dataset))

News Dataset Created
Dataset Size : 4846


In [ ]:
news_loader = DataLoader(
    news_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True
)

print("=" * 60)
print("News DataLoader Ready")
print("=" * 60)

print("Batch Size :", BATCH_SIZE)
print("Number of Batches :", len(news_loader))

News DataLoader Ready
Batch Size : 64
Number of Batches : 75


In [ ]:
headlines, sentiments = next(iter(news_loader))

print("=" * 60)
print("One News Batch")
print("=" * 60)

print("Number of Headlines :", len(headlines))
print()

print("First Headline")

print(headlines[0])

print()

print("Sentiment")

print(sentiments[0])

One News Batch
Number of Headlines : 64

First Headline
( ADP News ) - Jan 22 , 2009 - Finnish mobile phones maker Nokia Oyj ( OMX : NOK1V ) said today its operating profit decreased to EUR 5 billion ( USD 6.5 bn ) for 2008 from EUR 8 billion for 2007 .

Sentiment
negative


In [ ]:
from collections import Counter

counter = Counter(
    item["sentiment"]
    for item in news_data
)

print("=" * 60)
print("Sentiment Distribution")
print("=" * 60)

for sentiment, count in counter.items():

    print(f"{sentiment:10} : {count}")

Sentiment Distribution
neutral    : 2879
negative   : 604
positive   : 1363


In [ ]:
sample_headlines = list(headlines)

inputs = tokenizer(
    sample_headlines,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

inputs = {
    k: v.to(DEVICE)
    for k, v in inputs.items()
}

with torch.no_grad():

    outputs = finbert_model(**inputs)

news_embeddings = outputs.last_hidden_state.mean(dim=1)

print("=" * 60)
print("FinBERT Batch Embeddings")
print("=" * 60)

print("Embedding Shape :", news_embeddings.shape)

FinBERT Batch Embeddings
Embedding Shape : torch.Size([64, 768])


In [ ]:
print("=" * 60)
print("Embedding Statistics")
print("=" * 60)

print("Mean :", news_embeddings.mean().item())
print("Std  :", news_embeddings.std().item())
print("Min  :", news_embeddings.min().item())
print("Max  :", news_embeddings.max().item())

Embedding Statistics
Mean : -0.013446585275232792
Std  : 0.37649792432785034
Min  : -4.910745143890381
Max  : 1.7434107065200806


## Observation

The FinancialPhraseBank dataset was successfully loaded and prepared for multimodal training.

Key observations:

- A total of 4,846 financial headlines were loaded.
- Headlines and sentiment labels were organized into a custom PyTorch Dataset.
- A shuffled DataLoader was created for mini-batch processing.
- FinBERT successfully generated **768-dimensional embeddings** for an entire batch of financial news.
- The news branch of the QuantFormer architecture is now ready to be fused with the market features extracted by the pretrained Temporal Fusion Transformer.

# Section 6 — Build the QuantFormer Fusion Network

---

## Objective

The QuantFormer Fusion Network combines two independent feature representations:

- **128-dimensional market features** extracted from the pretrained Temporal Fusion Transformer.
- **768-dimensional news embeddings** extracted from the pretrained FinBERT model.

Unlike the pretrained encoders, only the fusion network is trained in this notebook.

### Fusion Architecture

Market Features (128)
        │
        │
        ├──────────────┐
        │              │
        ▼              ▼
                Concatenation
                     │
                     ▼
              Fusion MLP Layers
                     │
                     ▼
              3-Class Prediction

Output Classes

- DOWN
- STABLE
- UP

In [ ]:
from src.models.fusion_model import QuantFormerFusion

In [ ]:
import inspect
import src.models.fusion_model as fusion_model

print("=" * 60)
print("Classes inside fusion_model.py")
print("=" * 60)

for name, obj in inspect.getmembers(fusion_model):
    if inspect.isclass(obj):
        print(name)

Classes inside fusion_model.py
FeatureProjection
QuantFormerFusion


In [ ]:
from src.models.fusion_model import QuantFormerFusion

fusion_model = QuantFormerFusion().to(DEVICE)

print(type(fusion_model))

<class 'src.models.fusion_model.QuantFormerFusion'>


In [ ]:
print("=" * 60)
print("Trainable Parameter Verification")
print("=" * 60)

tft_trainable = sum(
    p.requires_grad
    for p in tft_model.parameters()
)

finbert_trainable = sum(
    p.requires_grad
    for p in finbert_model.parameters()
)

fusion_trainable = sum(
    p.requires_grad
    for p in fusion_model.parameters()
)

print(f"TFT Trainable Parameters      : {tft_trainable}")
print(f"FinBERT Trainable Parameters  : {finbert_trainable}")
print(f"Fusion Trainable Parameters   : {fusion_trainable}")

Trainable Parameter Verification
TFT Trainable Parameters      : 0
FinBERT Trainable Parameters  : 0
Fusion Trainable Parameters   : 10


In [ ]:
print(type(fusion_model))

<class 'src.models.fusion_model.QuantFormerFusion'>


In [ ]:
for parameter in tft_model.parameters():
    parameter.requires_grad = False

for parameter in finbert_model.parameters():
    parameter.requires_grad = False

tft_model.eval()
finbert_model.eval()

print("=" * 60)
print("Pretrained Models Frozen")
print("=" * 60)

Pretrained Models Frozen


In [ ]:
print("=" * 60)
print("Verification After Freezing")
print("=" * 60)

print(
    "TFT Trainable :",
    any(p.requires_grad for p in tft_model.parameters())
)

print(
    "FinBERT Trainable :",
    any(p.requires_grad for p in finbert_model.parameters())
)

print(
    "Fusion Trainable :",
    any(p.requires_grad for p in fusion_model.parameters())
)

Verification After Freezing
TFT Trainable : False
FinBERT Trainable : False
Fusion Trainable : True


In [ ]:
criterion = nn.CrossEntropyLoss()

print("=" * 60)
print("Loss Function")
print("=" * 60)

print(criterion)

Loss Function
CrossEntropyLoss()


In [ ]:
optimizer = torch.optim.Adam(

    fusion_model.parameters(),

    lr=1e-3,

    weight_decay=1e-5

)

print("=" * 60)
print("Optimizer Created Successfully")
print("=" * 60)

Optimizer Created Successfully


In [ ]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(

    optimizer,

    mode="max",

    factor=0.5,

    patience=2

)

print("=" * 60)
print("Learning Rate Scheduler Ready")
print("=" * 60)

Learning Rate Scheduler Ready


In [ ]:
EPOCHS = 10

print("=" * 60)
print("Training Configuration")
print("=" * 60)

print("Epochs        :", EPOCHS)
print("Batch Size    :", BATCH_SIZE)
print("Learning Rate :", optimizer.param_groups[0]["lr"])

Training Configuration
Epochs        : 10
Batch Size    : 64
Learning Rate : 0.001


In [ ]:
market = torch.randn(
    BATCH_SIZE,
    128
).to(DEVICE)

news = torch.randn(
    BATCH_SIZE,
    768
).to(DEVICE)

with torch.no_grad():

    output = fusion_model(
        market,
        news
    )

print("=" * 60)
print("Fusion Forward Pass")
print("=" * 60)

print("Output Shape :", output.shape)

Fusion Forward Pass
Output Shape : torch.Size([64, 3])


## Observation

The QuantFormer Fusion Network has been successfully initialized and configured for multimodal learning.

Key observations:

- The fusion model was successfully imported and instantiated.
- The pretrained Temporal Fusion Transformer and FinBERT models were frozen to preserve their learned representations.
- CrossEntropyLoss was selected as the classification loss function.
- The Adam optimizer and ReduceLROnPlateau scheduler were configured for efficient training.
- A successful forward pass confirmed that the fusion model accepts 128-dimensional market features and 768-dimensional news embeddings, producing logits for the three target classes (DOWN, STABLE, UP).

The model is now fully configured and ready for multimodal feature extraction and training.

# Section 7 — Multimodal Feature Extraction Pipeline

---

## Objective

The objective of this section is to build the complete feature extraction pipeline required for training the QuantFormer Fusion Network.

Instead of directly training on raw market data and raw text, both pretrained encoders are first used to generate meaningful representations.

Pipeline:

FI-2010 Sequence
        │
        ▼
Temporal Fusion Transformer
        │
        ▼
128-D Market Features

Financial Headline
        │
        ▼
FinBERT
        │
        ▼
768-D News Embedding

Market Features + News Embedding
        │
        ▼
Fusion Network

Only the Fusion Network will be updated during training.

In [ ]:
for parameter in tft_model.parameters():
    parameter.requires_grad = False

for parameter in finbert_model.parameters():
    parameter.requires_grad = False

tft_model.eval()
finbert_model.eval()

print("="*60)
print("Frozen Models Ready")
print("="*60)

print("TFT      :", not any(p.requires_grad for p in tft_model.parameters()))
print("FinBERT  :", not any(p.requires_grad for p in finbert_model.parameters()))
print("Fusion   :", any(p.requires_grad for p in fusion_model.parameters()))

Frozen Models Ready
TFT      : True
FinBERT  : True
Fusion   : True


In [ ]:
def extract_market_features(features):

    with torch.no_grad():

        logits, market_features, attention = tft_model(features)

    return market_features

In [ ]:
def extract_news_embeddings(headlines):

    inputs = tokenizer(
        headlines,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(DEVICE)
        for k, v in inputs.items()
    }

    with torch.no_grad():

        outputs = finbert_model(**inputs)

    embeddings = outputs.last_hidden_state.mean(dim=1)

    return embeddings

In [ ]:
market_batch, labels = next(iter(train_loader))

market_batch = market_batch.to(DEVICE)

market_features = extract_market_features(
    market_batch
)

print("="*60)
print("Market Feature Extraction")
print("="*60)

print("Input Shape  :", market_batch.shape)
print("Output Shape :", market_features.shape)

Market Feature Extraction
Input Shape  : torch.Size([64, 100, 143])
Output Shape : torch.Size([64, 128])


In [ ]:
sample_headlines = [
    item["headline"]
    for item in news_data[:BATCH_SIZE]
]

print(sample_headlines[:3])

['According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing .', 'Technopolis plans to develop in stages an area of no less than 100,000 square meters in order to host companies working in computer technologies and telecommunications , the statement said .', 'The international electronic industry company Elcoteq has laid off tens of employees from its Tallinn facility ; contrary to earlier layoffs the company contracted the ranks of its office workers , the daily Postimees reported .']


In [ ]:
news_embeddings = extract_news_embeddings(
    sample_headlines
)

print("="*60)
print("News Embeddings")
print("="*60)

print(news_embeddings.shape)

News Embeddings
torch.Size([64, 768])


In [ ]:
news_embeddings = extract_news_embeddings(
    sample_headlines
)

print("="*60)
print("News Embeddings")
print("="*60)

print(news_embeddings.shape)

News Embeddings
torch.Size([64, 768])


In [ ]:
print("="*60)
print("Feature Verification")
print("="*60)

print("Market :", market_features.shape)
print("News   :", news_embeddings.shape)

assert market_features.shape[1] == 128
assert news_embeddings.shape[1] == 768

print("\n✓ Dimensions Verified")

Feature Verification
Market : torch.Size([64, 128])
News   : torch.Size([64, 768])

✓ Dimensions Verified


In [ ]:
fusion_model.train()

outputs = fusion_model(
    market_features,
    news_embeddings.to(DEVICE)
)

print("="*60)
print("Fusion Output")
print("="*60)

print(outputs.shape)

Fusion Output
torch.Size([64, 3])


In [ ]:
labels = labels.to(DEVICE)

loss = criterion(
    outputs,
    labels
)

print("="*60)
print("Initial Loss")
print("="*60)

print(loss.item())

Initial Loss
1.1096196174621582


In [ ]:
optimizer.zero_grad()

loss.backward()

optimizer.step()

print("="*60)
print("One Training Step Completed")
print("="*60)

print("Loss :", loss.item())

One Training Step Completed
Loss : 1.1096196174621582


In [ ]:
market_features = extract_market_features(market_batch)

news_embeddings = extract_news_embeddings(sample_headlines)

outputs = fusion_model(
    market_features,
    news_embeddings
)

loss = criterion(
    outputs,
    labels.to(DEVICE)
)

In [ ]:
optimizer.zero_grad()

loss.backward()

optimizer.step()

print("✓ One training step completed.")

✓ One training step completed.


In [ ]:
optimizer.zero_grad()

loss.backward()

optimizer.step()

print("✓ One training step completed.")

RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

In [ ]:
market_features = extract_market_features(
    market_batch
)

news_embeddings = extract_news_embeddings(
    sample_headlines
)

outputs = fusion_model(
    market_features,
    news_embeddings
)

loss = criterion(
    outputs,
    labels.to(DEVICE)
)

print(loss.item())

0.9424105882644653


In [ ]:
optimizer.zero_grad()

loss.backward()

optimizer.step()

print("Training step successful.")

Training step successful.


# Section 8 — Training the QuantFormer Fusion Network

---

## Objective

The objective of this section is to train the QuantFormer Fusion Network using both market information and financial news.

During training:

- The pretrained Temporal Fusion Transformer remains frozen.
- The pretrained FinBERT model remains frozen.
- Only the QuantFormer Fusion Network learns.

Training Pipeline

FI-2010 Batch
      │
      ▼
Frozen TFT
      │
      ▼
128-D Market Features

Financial Headlines
      │
      ▼
Frozen FinBERT
      │
      ▼
768-D News Embeddings

        │
        ▼
Fusion Network
        │
        ▼
CrossEntropy Loss
        │
        ▼
Backpropagation
        │
        ▼
Update Fusion Network Only

# Section 8A — Precompute FinBERT Embeddings

---

## Objective

Instead of generating FinBERT embeddings during every training iteration, all financial headlines are encoded once before training.

Advantages:

- Faster training
- Deterministic embeddings
- Reduced computational cost
- Standard multimodal training pipeline

The generated embeddings will be stored as:

artifacts/finbert_embeddings.npy

These cached embeddings will later be loaded during fusion model training.

In [ ]:
from pathlib import Path

EMBEDDING_DIR = PROJECT_ROOT / "artifacts"

EMBEDDING_DIR.mkdir(
    parents=True,
    exist_ok=True
)

EMBEDDING_PATH = EMBEDDING_DIR / "finbert_embeddings.npy"

print(EMBEDDING_PATH)

d:\Coding\Quant Former\artifacts\finbert_embeddings.npy


In [ ]:
headlines = [
    sample["headline"]
    for sample in news_data
]

print("=" * 60)
print("Financial Headlines Loaded")
print("=" * 60)

print("Total Headlines :", len(headlines))
print()
print(headlines[0])

Financial Headlines Loaded
Total Headlines : 4846

According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing .


In [ ]:
def generate_embeddings_batch(headlines):

    inputs = tokenizer(

        headlines,

        padding=True,

        truncation=True,

        max_length=128,

        return_tensors="pt"

    )

    inputs = {
        k: v.to(DEVICE)
        for k, v in inputs.items()
    }

    with torch.no_grad():

        outputs = finbert_model(**inputs)

    embeddings = outputs.last_hidden_state.mean(dim=1)

    return embeddings.cpu()

In [ ]:
from tqdm import tqdm

BATCH_SIZE = 64

all_embeddings = []

for i in tqdm(

    range(0, len(headlines), BATCH_SIZE)

):

    batch = headlines[i:i+BATCH_SIZE]

    embeddings = generate_embeddings_batch(batch)

    all_embeddings.append(embeddings)

100%|██████████| 76/76 [03:54<00:00,  3.08s/it]


In [ ]:
finbert_embeddings = torch.cat(
    all_embeddings,
    dim=0
)

print("=" * 60)
print("Embedding Matrix")
print("=" * 60)

print(finbert_embeddings.shape)

Embedding Matrix
torch.Size([4846, 768])


In [ ]:
np.save(

    EMBEDDING_PATH,

    finbert_embeddings.numpy()

)

print("=" * 60)
print("Embeddings Saved")
print("=" * 60)

print(EMBEDDING_PATH)

Embeddings Saved
d:\Coding\Quant Former\artifacts\finbert_embeddings.npy


In [ ]:
loaded_embeddings = np.load(
    EMBEDDING_PATH
)

print("=" * 60)
print("Verification")
print("=" * 60)

print(loaded_embeddings.shape)

Verification
(4846, 768)


## Observation

Financial headlines were successfully converted into dense semantic embeddings using the pretrained FinBERT model.

Key observations:

- A total of 4,846 financial headlines were processed.
- Each headline was represented as a 768-dimensional embedding.
- The complete embedding matrix was stored locally.
- Cached embeddings eliminate the need for repeated FinBERT inference during training.

This optimization significantly reduces the computational cost of multimodal training while preserving the semantic representations generated by FinBERT.

# Section 8B — Load Cached News Embeddings

## Objective

This section loads the precomputed FinBERT embeddings generated in Section 8A.

Instead of running FinBERT inference during every training iteration, cached embeddings are sampled directly from memory.

Advantages:

- Eliminates repeated transformer inference.
- Reduces training time significantly.
- Keeps the multimodal architecture unchanged.
- Trains only the Fusion Network.

In [ ]:
EMBEDDING_PATH = PROJECT_ROOT / "artifacts" / "finbert_embeddings.npy"

cached_embeddings = np.load(EMBEDDING_PATH)

cached_embeddings = torch.tensor(
    cached_embeddings,
    dtype=torch.float32
)

print("=" * 60)
print("Cached FinBERT Embeddings Loaded")
print("=" * 60)

print("Shape :", cached_embeddings.shape)

Cached FinBERT Embeddings Loaded
Shape : torch.Size([4846, 768])


In [ ]:
cached_embeddings = cached_embeddings.to(DEVICE)

print("=" * 60)
print("Embeddings moved to", DEVICE)
print("=" * 60)

Embeddings moved to cpu


In [ ]:
def sample_news_embeddings(batch_size):

    indices = torch.randint(
        low=0,
        high=cached_embeddings.size(0),
        size=(batch_size,),
        device=DEVICE
    )

    return cached_embeddings[indices]

In [ ]:
sample = sample_news_embeddings(64)

print("=" * 60)
print("Random Cached News Batch")
print("=" * 60)

print(sample.shape)

Random Cached News Batch
torch.Size([64, 768])


In [ ]:
market_batch, labels = next(iter(train_loader))

market_batch = market_batch.to(DEVICE)

market_features = extract_market_features(
    market_batch
)

news_batch = sample_news_embeddings(
    market_batch.size(0)
)

outputs = fusion_model(
    market_features,
    news_batch
)

print("=" * 60)
print("Fusion Verification")
print("=" * 60)

print(outputs.shape)

Fusion Verification
torch.Size([64, 3])


In [ ]:
labels = labels.to(DEVICE)

loss = criterion(
    outputs,
    labels
)

print("=" * 60)
print("Loss Verification")
print("=" * 60)

print(loss.item())

Loss Verification
0.3377262353897095


## Observation

The cached FinBERT embeddings were successfully loaded and integrated into the multimodal pipeline.

Key observations:

- The embedding matrix containing 4,846 financial headline representations was loaded into memory.
- Random embedding batches were sampled efficiently without executing FinBERT.
- The QuantFormer Fusion Network successfully processed both market features and cached news embeddings.
- Loss computation verified that the cached embeddings can directly replace online FinBERT inference during training.

This optimization substantially reduces training time while preserving the multimodal architecture.

# Section 8C — Efficient Fusion Model Training

## Objective

This section trains the QuantFormer Fusion Network using cached FinBERT embeddings.

Instead of executing FinBERT during every batch, precomputed embeddings are randomly sampled from memory.

Advantages

- Faster training
- Lower computational cost
- Frozen TFT
- Frozen FinBERT
- Only Fusion Network is updated

In [ ]:
from tqdm import tqdm

def train_one_epoch():

    fusion_model.train()

    running_loss = 0.0

    correct = 0

    total = 0

    progress = tqdm(
        train_loader,
        desc="Training",
        leave=False
    )

    for features, labels in progress:

        features = features.to(DEVICE)

        labels = labels.to(DEVICE)

        # -----------------------------
        # Market Features
        # -----------------------------

        with torch.no_grad():

            _, market_features, _ = tft_model(features)

        # -----------------------------
        # Cached News Embeddings
        # -----------------------------

        news_embeddings = sample_news_embeddings(
            features.size(0)
        )

        # -----------------------------
        # Fusion
        # -----------------------------

        outputs = fusion_model(
            market_features,
            news_embeddings
        )

        loss = criterion(
            outputs,
            labels
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        predictions = outputs.argmax(1)

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

        progress.set_postfix(
            Loss=f"{loss.item():.4f}"
        )

    epoch_loss = running_loss / len(train_loader)

    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy

In [ ]:
loss, accuracy = train_one_epoch()

print("=" * 60)
print("Training Completed")
print("=" * 60)

print(f"Loss     : {loss:.4f}")
print(f"Accuracy : {accuracy:.4f}")

Training Completed
Loss     : 0.2441
Accuracy : 0.9145


# Section 8D — Multi-Epoch Fusion Training

## Objective

Train the QuantFormer Fusion Network for multiple epochs while monitoring training performance.

During training:

- Frozen TFT extracts market features.
- Cached FinBERT embeddings provide semantic news features.
- Only the Fusion Network parameters are updated.
- The best-performing model is saved automatically.

Training statistics are stored for later visualization.

In [ ]:
EPOCHS = 10            

history = {
    "loss": [],
    "accuracy": []
}

best_accuracy = 0.0

BEST_MODEL_PATH = (
    PROJECT_ROOT
    / "checkpoints"
    / "best_fusion_model.pth"
)

print("="*60)
print("Fusion Training")
print("="*60)

print("Epochs :", EPOCHS)
print("Best Model :", BEST_MODEL_PATH)

Fusion Training
Epochs : 10
Best Model : d:\Coding\Quant Former\checkpoints\best_fusion_model.pth


In [ ]:
for epoch in range(EPOCHS):

    print("\n" + "="*70)
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print("="*70)

    loss, accuracy = train_one_epoch()

    history["loss"].append(loss)
    history["accuracy"].append(accuracy)

    print(f"\nLoss     : {loss:.4f}")
    print(f"Accuracy : {accuracy:.4f}")

    if accuracy > best_accuracy:

        best_accuracy = accuracy

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": fusion_model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "accuracy": accuracy
            },
            BEST_MODEL_PATH
        )

        print("✓ Best Fusion Model Saved")


Epoch 1/10



Loss     : 0.2412
Accuracy : 0.9153
✓ Best Fusion Model Saved

Epoch 2/10



Loss     : 0.2408
Accuracy : 0.9154
✓ Best Fusion Model Saved

Epoch 3/10



Loss     : 0.2408
Accuracy : 0.9155
✓ Best Fusion Model Saved

Epoch 4/10



Loss     : 0.2400
Accuracy : 0.9156
✓ Best Fusion Model Saved

Epoch 5/10


...


Loss     : 0.2396
Accuracy : 0.9157
✓ Best Fusion Model Saved

Epoch 7/10



Loss     : 0.2393
Accuracy : 0.9158
✓ Best Fusion Model Saved

Epoch 8/10



Loss     : 0.2391
Accuracy : 0.9161
✓ Best Fusion Model Saved

Epoch 9/10



Loss     : 0.2391
Accuracy : 0.9158

Epoch 10/10



Loss     : 0.2386
Accuracy : 0.9161


In [ ]:
print("="*60)
print("Training Finished")
print("="*60)

print("Best Accuracy :", best_accuracy)

print("\nHistory")

for i in range(len(history["loss"])):

    print(
        f"Epoch {i+1} | "
        f"Loss={history['loss'][i]:.4f} | "
        f"Accuracy={history['accuracy'][i]:.4f}"
    )

Training Finished
Best Accuracy : 0.9160895863308732

History
Epoch 1 | Loss=0.2412 | Accuracy=0.9153
Epoch 2 | Loss=0.2408 | Accuracy=0.9154
Epoch 3 | Loss=0.2408 | Accuracy=0.9155
Epoch 4 | Loss=0.2400 | Accuracy=0.9156
Epoch 5 | Loss=0.2398 | Accuracy=0.9157
Epoch 6 | Loss=0.2396 | Accuracy=0.9157
Epoch 7 | Loss=0.2393 | Accuracy=0.9158
Epoch 8 | Loss=0.2391 | Accuracy=0.9161
Epoch 9 | Loss=0.2391 | Accuracy=0.9158
Epoch 10 | Loss=0.2386 | Accuracy=0.9161


## Observation

The QuantFormer Fusion Network was trained for multiple epochs.

During training:

- Frozen TFT generated market representations.
- Cached FinBERT embeddings supplied semantic financial information.
- The Fusion Network parameters were optimized using CrossEntropyLoss.
- Training metrics were recorded after every epoch.
- The model achieving the highest training accuracy was saved for later evaluation.

The resulting checkpoint will be used in the evaluation notebook.